[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pflahert/hd-stats-workshop/blob/main/notebooks/homework.ipynb)

# Homework: Day 1 → Day 2
**Due at the start of Day 2** | Estimated time: 1.5 hours

This homework consolidates Day 1 material and bridges to Day 2. It maps onto the new Day-1 structure:

- **Q1 — Multiple Testing Concepts (LO1)**. Exercises **Lecture 1** (hypothesis testing: Bonferroni / FWER, BH / FDR, $q$-values).
- **Q2 — FDR in Practice (LO1, LO3)**. Connects both Day-1 lectures: BH-adjusted $p$-values from **Lecture 1**; `limma`'s empirical-Bayes variance shrinkage and $\hat\pi_0$ estimation from **Lecture 2**.
- **Q3 — Evaluating Analysis Output (LO3)**. Exercises the critical-evaluation thread from both lectures and from Labs 1 and 2.

You are welcome to use AI for coding tasks. Conceptual and interpretation questions must reflect your own understanding.

> **⚠ This notebook uses the R kernel** (Q2 requires `limma` and `qvalue`, which are R/Bioconductor packages).
> **In Colab:** Click **Runtime → Change runtime type → R**, then re-open this notebook (or refresh).
> Once the R kernel is selected, the cell below should print "R version 4.x.x".

In [ ]:
# Verify you are running the R kernel
cat("R version:", R.version.string, "\n")
cat("Working directory:", getwd(), "\n")


---
## Q1: Multiple Testing Concepts (LO1)

A proteomics experiment measures the abundance of $p = 5{,}000$ proteins in $n = 30$ tissue samples (15 tumor, 15 normal). For each protein, a two-sample $t$-test is performed.

### (a)
Under the global null (no protein is differentially abundant), how many proteins would you expect to reject at $\alpha = 0.05$ without any correction?

**YOUR ANSWER:**


### (b)
The Bonferroni-corrected threshold is $\alpha / m$. Compute this threshold. A protein with a "real" effect has a $p$-value of $3 \times 10^{-4}$. Would it survive Bonferroni correction?

**YOUR ANSWER:**



In [ ]:
# You can verify (a) and (b) here:
m <- 5000
alpha <- 0.05
expected_fp_uncorrected <- m * alpha
bonf_threshold <- alpha / m
cat("Expected false positives (uncorrected):", expected_fp_uncorrected, "\n")
cat("Bonferroni threshold:", bonf_threshold, "\n")
cat("Real effect p = 3e-4 survives Bonferroni?", 3e-4 < bonf_threshold, "\n")


### (c)
You apply the BH procedure at FDR $= 0.10$ and get 120 rejections. Give an interpretation of this result in plain language that a biologist would understand.

**YOUR ANSWER:**


### (d)
Your collaborator says "We found 120 significant proteins." A reviewer replies "But 12 of those are probably false positives!" Is the reviewer being unreasonable? Explain.

**YOUR ANSWER:**



---
## Q2: FDR in Practice (LO1, LO3)

Using the Golub dataset (introduced in **Lecture 1** and used in **Labs 1 and 2**), perform the following analysis. You should use AI to write the R code, but evaluate its correctness and interpret the results yourself.

The two empirical-Bayes ideas you will exercise here:

- **From Lecture 2**: `limma`'s `eBayes()` shrinks gene-level variance estimates toward a common prior, producing more stable moderated $t$-statistics than raw $t$-tests.
- **From Lecture 1**: BH adjustment converts $p$-values to FDR-controlled discoveries. Storey's $\hat\pi_0$ from the same lecture provides an adaptive variant via the `qvalue` package.

### Setup — load the Golub data and install required packages

In [ ]:
# Install limma and qvalue from Bioconductor (takes a few minutes in Colab)
if (!require("BiocManager", quietly = TRUE)) {
  install.packages("BiocManager", quiet = TRUE)
}
BiocManager::install(c("limma", "qvalue"), update = FALSE, ask = FALSE, quiet = TRUE)

library(limma)
library(qvalue)
cat("limma version:", as.character(packageVersion("limma")), "\n")
cat("qvalue version:", as.character(packageVersion("qvalue")), "\n")


In [ ]:
# Load the Golub expression matrix and metadata directly from the workshop repo
expr_url <- "https://raw.githubusercontent.com/pflahert/hd-stats-workshop/main/data/golub_expression.csv"
meta_url <- "https://raw.githubusercontent.com/pflahert/hd-stats-workshop/main/data/golub_metadata.csv"

expr <- read.csv(expr_url, row.names = 1, check.names = FALSE)
meta <- read.csv(meta_url)

cat("Expression matrix:", nrow(expr), "genes x", ncol(expr), "samples\n")
cat("Metadata rows:", nrow(meta), "\n")
cat("Class distribution:\n")
print(table(meta$subtype))

# limma expects genes as rows, samples as columns -- already in that orientation
# Sanity check column names match metadata sample IDs
stopifnot(all(colnames(expr) == meta$sample_id))


### (a) BH-adjusted p-values via limma

Direct an AI assistant to compute BH-adjusted *p*-values for the ALL vs. AML comparison **using `limma`** (Lecture 2's empirical-Bayes variance shrinkage) rather than raw $t$-tests. Report the number of genes significant at FDR $< 0.01$, $< 0.05$, $< 0.10$ (BH thresholding is Lecture 1).

**Include the prompt you used and note whether the AI spontaneously chose `limma` or whether you had to direct it.**

**Suggested prompt** (paste-ready, with R-specific format directive):

> "Reply with a single R code block (no prose, no .docx file — just runnable code I can paste into an R-kernel Colab cell).
>
> The Golub leukemia microarray dataset has been loaded into an R data frame `expr` with 3,051 genes (rows) × 72 samples (columns), and a data frame `meta` with columns `sample_id` and `subtype` (one of `'ALL'`, `'AML'`). Using `limma`, fit a linear model with empirical-Bayes variance shrinkage for the ALL vs. AML contrast, retrieve the BH-adjusted p-values from `topTable(..., adjust.method = 'BH', number = Inf)`, and report the number of genes with `adj.P.Val` below 0.01, 0.05, and 0.10. Print only the counts, not the full table."

**Why the format directive matters.** The UMass campus GenAI platform (LibreChat) returns long answers as `.docx` files unless told otherwise. The format prefix above keeps the output paste-ready.

**Your prompt to the AI:**


**The AI you used:** (e.g., the [UMass campus GenAI platform](https://www.umass.edu/it/genai/platform), ChatGPT, Claude)


**Did the AI spontaneously choose `limma`, or did you have to direct it?**

In [ ]:
# Paste / refine the AI-generated limma code here.
# Reference scaffold (you should still verify each step):
#   1. Build a design matrix from meta$subtype (factor with levels ALL, AML)
#   2. Fit a linear model with lmFit()
#   3. Apply empirical Bayes shrinkage with eBayes()
#   4. Use topTable(..., adjust.method = "BH", number = Inf) to get BH-adjusted p-values
#   5. Count genes with adj.P.Val below 0.01, 0.05, 0.10

# YOUR CODE HERE


### (b) Estimate $\hat\pi_0$ with qvalue

Use the `qvalue` package to estimate $\hat{\pi}_0$. What does this estimate tell you about the dataset?


In [ ]:
# YOUR CODE HERE
# Hint: qvalue(p) where p is the vector of raw p-values from limma's topTable
# Then inspect the qvalue object's pi0 component.


**Interpretation of $\hat\pi_0$:**



### (c) P-value histogram with annotation

Make a *p*-value histogram. Annotate it: mark the region where you think the true alternatives are concentrated, and the region dominated by true nulls.


In [ ]:
# YOUR CODE HERE
# A simple base-R histogram with abline() marking the flat-region threshold is fine.
# Or use ggplot2 with geom_histogram() and annotate().


**One-paragraph interpretation of (a)–(c):**

(Discuss: How many discoveries at each FDR level? What does $\hat\pi_0$ suggest about the proportion of nulls? Does the histogram shape match your expectation? What was AI-generated vs. your own?)




---
## Q3: Evaluating Analysis Output (LO3)

### (a) Two specifications for the same task

Write **two** specifications for the same analysis task: *"Perform differential expression analysis on a microarray dataset comparing treatment and control groups."*

- **Specification A** (vague, likely to produce incorrect/incomplete analysis):


- **Specification B** (precise, likely to produce correct analysis on the first attempt — name the method, the test, the multiple-testing correction, and the threshold):


**What makes Specification B better?**


### (b) Verifying a citation

You read (in a paper or AI output) the claim:
> *"Gene XYZ promotes cell proliferation in breast cancer through the PI3K/AKT pathway (Smith et al., 2019)."*

Describe the steps you would take to verify this before including it in your own work.

**YOUR ANSWER:**


### (c) An error you encountered today

Describe one specific error you encountered during today's labs — whether in AI-generated code, your own code, or a collaborator's analysis.

- What was the error?
- How did you catch it?
- What statistical knowledge was required to recognize it?

**Suggested AI prompt for follow-up** (paste-ready, with format directive):

> "Reply with a single R code block (no prose, no .docx file — just runnable code I can paste into an R-kernel Colab cell). Here is a snippet of analysis code: \[paste your snippet\]. Identify the most likely statistical error and write a corrected version of the snippet. If the original is correct, return the same code unchanged with a one-line comment confirming this."

**YOUR ANSWER:**


### (d) A short review

You receive a complete differential expression analysis from a collaborator. It uses raw $t$-tests with BH correction on a microarray dataset with 72 samples. Write a short review (3–4 sentences) describing what's adequate and what should be changed, and why.

*(Hint: connect this to Lecture 2's argument that `limma`'s empirical-Bayes variance shrinkage is preferred over raw $t$-tests when $n$ is small. The [Decision Guide](https://pflahert.github.io/hd-stats-workshop/syllabus/decision-guide.html) summarizes the recommended methods.)*

**YOUR REVIEW:**

---
## Submission

Bring your completed notebook (or written answers + code outputs for Q2) to Day 2. Be prepared to reference your answers during Lecture 3 and Lecture 4 discussions.

### Assessment criteria

This is a formative assessment. Completeness and engagement matter more than getting every detail right. We are particularly interested in your interpretation of statistical results in your own words and your ability to identify errors in analysis code or output.

> **Tip:** `File → Download → Download .ipynb` to keep a local copy of your work.